In [57]:
import os
import numpy as np
import h5py
from skimage.metrics import structural_similarity as ssim

In [ ]:
from matplotlib import pyplot as plt

def visualize_results(input, gt, pred):
  """
  input, gt, pred: [d, w, h]
  """
  slice = 40
  fig, axes = plt.subplots(1, 3, figsize=(15, 10))
  # vmin = (0+1000)/4096
  # vmax = (200+1000)/4096
  vmin = 0
  vmax = 1
  axes[0].imshow(input[slice], cmap='gray', vmin=0, vmax=1)
  axes[0].set_title('Input')
  axes[1].imshow(gt[slice], cmap='gray', vmin=vmin, vmax=vmax)
  axes[1].set_title('Target')
  axes[2].imshow(pred[slice], cmap='gray', vmin=vmin, vmax=vmax)
  axes[2].set_title('Prediction')

  for ax in axes: ax.axis('off')
  plt.tight_layout()
  plt.show()
  plt.close(fig)


In [ ]:
## all test set
output_dir = 'DDPM/output/test/result_2d_epoch_1274'
categories = ['brain', 'pelvis', 'AB', 'HN', 'TH']
cat_mae = {cat: [] for cat in categories}
cat_ssim = {cat: [] for cat in categories}
all_mae = []
all_ssim = []

for i, file in enumerate(os.listdir(output_dir)):
    if file.endswith('.hdf5'):
        for cat in categories:
            if cat in file:
                with h5py.File(os.path.join(output_dir, file), 'r') as f:
                    pred = f['output'][()]
                    gt = f['target'][()]
                    input = f['input'][()]
                    mask = f['input_mask'][()]
                    #pred *= mask
                    mae = np.mean(np.abs(pred - gt))
                    ssim_value = ssim(pred, gt, data_range=1., channel_axis=None)
                    cat_mae[cat].append(mae)
                    cat_ssim[cat].append(ssim_value)
                    all_mae.append(mae)
                    all_ssim.append(ssim_value)
                break

for cat in categories:
    if cat_mae[cat]:
        print(cat)
        print(f"MAE: {[round(x, 3) for x in cat_mae[cat]]}")
        print(f"SSIM: {[round(x, 3) for x in cat_ssim[cat]]}")
        print(f"avg MAE = {np.mean(cat_mae[cat]):.5f}, avg SSIM = {np.mean(cat_ssim[cat]):.5f}, count = {len(cat_mae[cat])}")

print('avg MAE = ', np.mean(all_mae), 'avg SSIM = ', np.mean(all_ssim))


brain: avg MAE = 0.01626, avg SSIM = 0.83348, count = 65
pelvis: avg MAE = 0.02094, avg SSIM = 0.77745, count = 55
AB: avg MAE = 0.01841, avg SSIM = 0.72685, count = 47
HN: avg MAE = 0.01636, avg SSIM = 0.79844, count = 70
TH: avg MAE = 0.01639, avg SSIM = 0.74586, count = 76

 avg MAE =  0.017460287 avg SSIM =  0.7785118356865728
